In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Draw
import py3Dmol

def get_molecule_from_name(name):
    # Search for the compound by name
    compounds = pcp.get_compounds(name, 'name')
    if compounds:
        print(compounds[0].to_dict())
        smiles = compounds[0].isomeric_smiles
        molecule = Chem.MolFromSmiles(smiles)
        return molecule
    else:
        return None

def draw_2d_structure(molecule):
    # Draws 2D structure of the molecule
    img = Draw.MolToImage(molecule)
    return img

def show_3D_structure(molecule):
    # Visualize 3D structure
    mb = Chem.MolToMolBlock(molecule)
    viewer = py3Dmol.view(width=400, height=300)
    viewer.addModel(mb, 'mol')
    viewer.setStyle({'stick': {}})
    viewer.zoomTo()
    return viewer

# Example Usage
name = "PFHxS"
molecule = get_molecule_from_name(name)

if molecule:
    # Display the RDKit image
    img = draw_2d_structure(molecule)
    img.show()

    # Display 3D structure
    viewer = show_3D_structure(molecule)
    viewer.show()
else:
    print("No compound found with that name.")


In [8]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem import AllChem


progesterone_smiles = 'CC(=O)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CCC4=CC(=O)CC[C@]34C)C'
mol = Chem.MolFromSmiles(progesterone_smiles)

# Optional: set a title in the SDF
mol.SetProp('_Name', 'Progesterone')

# Generate 3D coordinates
mol3d = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol3d, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol3d)

# Write to SDF
writer = Chem.SDWriter('progesterone5.sdf')
writer.write(mol3d)
writer.close()


In [2]:
import os
import glob
import re
import pymol
from pymol import cmd

def natural_key(s):
    # Sort "docked", "docked2", ..., "docked10" in human/numeric order
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

def load_docked_poses_batch(
    protein,
    pose_pattern,
    ligand_name,
    output_dir=".",
    chain="C",
    resi_base=1,
    start_index=1,
    remove_waters=True
):
    """
    Load multiple multi-MODEL docked PDBs and save protein-ligand complexes with
    a global running index across files.

    Args:
        protein (str): Path to protein PDB file.
        pose_pattern (str): Glob pattern for pose files, e.g., "docked*.pdb".
        ligand_name (str): 3-letter ligand residue name to set (e.g., "HCY").
        output_dir (str): Where to save complex_###.pdb files.
        chain (str): Chain ID to assign to ligand.
        resi_base (int): Base for ligand residue numbering (resi_base + idx).
        start_index (int): First index for output numbering.
        remove_waters (bool): Remove crystallographic waters from protein.
    """
    cmd.reinitialize()
    cmd.load(protein, "prot")
    prot_name = protein.split(".")[0]
    if remove_waters:
        cmd.remove("resn HOH")
    cmd.set("pdb_conect_all", 1)  # ensure CONECT records for ligands
   

    pose_files = sorted(glob.glob(pose_pattern), key=natural_key)
    if not pose_files:
        print(f"No pose files matched pattern: {pose_pattern}")
        return

    os.makedirs(output_dir, exist_ok=True)

    idx = start_index
    for fidx, pose_file in enumerate(pose_files, start=1):
        obj_src = f"lig_src_{fidx}"
        cmd.load(pose_file, obj_src)  # multi-MODEL -> multi-state object

        n_states = cmd.count_states(obj_src)
        print(f"{pose_file}: {n_states} pose(s)")

        for state in range(1, n_states + 1):
            lig_obj = f"lig_{idx:03d}"
            cmd.create(lig_obj, obj_src, state, 1)  # extract single-state ligand
            # annotate ligand
            cmd.alter(lig_obj, f"resn='{ligand_name}'; chain='{chain}'; resi='{resi_base + idx}'")
            cmd.sort(lig_obj)

            complex_obj = f"{prot_name}_{ligand_name}_{idx:03d}" # output format e.g. "6ob5_HCY_001"
            cmd.create(complex_obj, f"prot or {lig_obj}")
            out_path = os.path.join(output_dir, f"{complex_obj}.pdb")
            cmd.save(out_path, complex_obj)
            print(f"  -> wrote {out_path}")

            # Clean up the temporary ligand object (keep complex)
            cmd.delete(lig_obj)
            idx += 1

        # remove the multi-state source object for this file
        cmd.delete(obj_src)

    print(f"Done. Wrote complexes up to index {idx-1:03d}.")

if __name__ == "__main__":
   
    pymol.finish_launching(['pymol', '-qc'])
   
    load_docked_poses_batch(
        protein="AcrR.pdb",
        pose_pattern="docked*.pdb",   # matches docked.pdb, docked2.pdb, ... docked5.pdb
        ligand_name="STR",
        output_dir=".",
        chain="C",
        resi_base=1,
        start_index=1,
        remove_waters=True
    )

docked2.pdb: 9 pose(s)
  -> wrote ./AcrR_STR_001.pdb
  -> wrote ./AcrR_STR_002.pdb
  -> wrote ./AcrR_STR_003.pdb
  -> wrote ./AcrR_STR_004.pdb
  -> wrote ./AcrR_STR_005.pdb
  -> wrote ./AcrR_STR_006.pdb
  -> wrote ./AcrR_STR_007.pdb
  -> wrote ./AcrR_STR_008.pdb
  -> wrote ./AcrR_STR_009.pdb
docked3.pdb: 9 pose(s)
  -> wrote ./AcrR_STR_010.pdb
  -> wrote ./AcrR_STR_011.pdb
  -> wrote ./AcrR_STR_012.pdb
  -> wrote ./AcrR_STR_013.pdb
  -> wrote ./AcrR_STR_014.pdb
  -> wrote ./AcrR_STR_015.pdb
  -> wrote ./AcrR_STR_016.pdb
  -> wrote ./AcrR_STR_017.pdb
  -> wrote ./AcrR_STR_018.pdb
docked4.pdb: 9 pose(s)
  -> wrote ./AcrR_STR_019.pdb
  -> wrote ./AcrR_STR_020.pdb
  -> wrote ./AcrR_STR_021.pdb
  -> wrote ./AcrR_STR_022.pdb
  -> wrote ./AcrR_STR_023.pdb
  -> wrote ./AcrR_STR_024.pdb
  -> wrote ./AcrR_STR_025.pdb
  -> wrote ./AcrR_STR_026.pdb
  -> wrote ./AcrR_STR_027.pdb
docked5.pdb: 9 pose(s)
  -> wrote ./AcrR_STR_028.pdb
  -> wrote ./AcrR_STR_029.pdb
  -> wrote ./AcrR_STR_030.pdb
  -> wro